# Workspace access → graph nodes/edges  (step 2)

Turns the `workspace_access` lakehouse table into the **same nodes/edges shape** as the
Entra security-group graph (`scripts/entra_groups_to_graph.py`), so the two load into
one SQL Server / Azure SQL graph and the correlation query works:

- **Workspace** nodes (`node_type='Workspace'`, `node_id` = workspace GUID)
- **HAS_ACCESS** edges: `principal -HAS_ACCESS-> workspace` (with `role`, `is_write_role`)
- **Principal** nodes (User / Group / ServicePrincipal) for anyone with access — these
  merge with the Entra nodes by `node_id` (Entra objectId) at load time.

Writes Delta tables `graph_access_nodes` / `graph_access_edges` (queryable in the
lakehouse) **and** single CSVs to `Files/graph/` for download into Azure SQL.

## Prerequisites
- `workspace_access_to_lakehouse` has run, so `workspace_access` exists.

In [ ]:
# ── Config & paths ───────────────────────────────────────────────────────────
from datetime import datetime, timezone
import notebookutils
from pyspark.sql import functions as F

LAKEHOUSE_NAME   = "lh_fabric_management"
LAKEHOUSE_SCHEMA = "fabricmanagement"
SRC              = "workspace_access"
TBL_NODES        = "graph_access_nodes"
TBL_EDGES        = "graph_access_edges"

RUN_TS = datetime.now(timezone.utc).replace(tzinfo=None, microsecond=0)
print("Run timestamp (UTC):", RUN_TS.isoformat())

def _lh_props(name):
    lh = notebookutils.lakehouse.get(name)
    return (lh.get("properties") or {}) if isinstance(lh, dict) else {}

_P = _lh_props(LAKEHOUSE_NAME)
TABLES_PATH = _P.get("oneLakeTablesPath") or f'{_P.get("abfsPath","").rstrip("/")}/Tables'
FILES_PATH  = _P.get("oneLakeFilesPath")  or f'{_P.get("abfsPath","").rstrip("/")}/Files'

def _table_uri(name):
    sub = name if not LAKEHOUSE_SCHEMA else f"{LAKEHOUSE_SCHEMA}/{name}"
    return f"{TABLES_PATH}/{sub}"

In [ ]:
# ── 1. Read the latest workspace_access snapshot ─────────────────────────────
wa_all = spark.read.format("delta").load(_table_uri(SRC))
latest = wa_all.agg(F.max("snapshot_date")).first()[0]
wa = wa_all.where(F.col("snapshot_date") == F.lit(latest))
print(f"workspace_access rows (snapshot {latest}): {wa.count()}")

has_pid = F.col("principal_id").isNotNull() & (F.col("principal_id") != "")

In [ ]:
# ── 2. Build nodes (Workspace + principals) and HAS_ACCESS edges ─────────────
ws_nodes = (wa.select(F.col("workspace_id").alias("node_id"),
                      F.lit("Workspace").alias("node_type"),
                      F.col("workspace_name").alias("display_name"),
                      F.lit(None).cast("string").alias("upn"),
                      F.lit(None).cast("string").alias("mail"))
              .where(F.col("workspace_id").isNotNull())
              .dropDuplicates(["node_id"]))

# principal nodes — merge with the Entra nodes by node_id (Entra objectId) in SQL
prin_nodes = (wa.where(has_pid)
                .select(F.col("principal_id").alias("node_id"),
                        F.col("principal_type").alias("node_type"),
                        F.col("display_name").alias("display_name"),
                        F.lit(None).cast("string").alias("upn"),
                        F.col("email").alias("mail"))
                .dropDuplicates(["node_id"]))

access_nodes = ws_nodes.unionByName(prin_nodes)

access_edges = (wa.where(has_pid)
                  .select(F.col("principal_id").alias("from_id"),
                          F.col("workspace_id").alias("to_id"),
                          F.lit("HAS_ACCESS").alias("edge_type"),
                          F.col("principal_type").alias("from_type"),
                          F.lit("Workspace").alias("to_type"),
                          F.col("role"),
                          F.col("is_write_role"))
                  .dropDuplicates(["from_id", "to_id"]))

print(f"access nodes: {access_nodes.count()}  (Workspace + principals)")
print(f"HAS_ACCESS edges: {access_edges.count()}")
access_nodes.groupBy("node_type").count().show(truncate=False)

In [ ]:
# ── 3. Write Delta tables + single CSVs to Files/graph/ ──────────────────────
def write_delta(df, name):
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(_table_uri(name))
    print(f"  {df.count():>6} rows -> {name} (delta)")

write_delta(access_nodes, TBL_NODES)
write_delta(access_edges, TBL_EDGES)

# single-file CSVs for download / Azure SQL load
def write_csv(df, sub):
    out = f"{FILES_PATH}/graph/{sub}"
    df.coalesce(1).write.mode("overwrite").option("header", "true").csv(out)
    print(f"  csv -> Files/graph/{sub}/ (one part-*.csv)")

write_csv(access_nodes, "access_nodes")
write_csv(access_edges, "access_edges")

## Load into Azure SQL

1. Download `Files/graph/access_nodes/part-*.csv` and `.../access_edges/part-*.csv`
   (rename to `access_nodes.csv` / `access_edges.csv`).
2. Load all four CSVs — the Entra `graph_nodes.csv` / `graph_edges.csv` and these two —
   using **`sql/entra_graph.sql`**. Node inserts use `WHERE NOT EXISTS`, so the Entra and
   access node sets union on `node_id` (Entra objectId) without duplicates.
3. Run the correlation query at the bottom of that script to find direct user→workspace
   grants already covered by a group the user belongs to.